# LightGBM

**Highly efficient tree based boosting ml algorithm designed specifically for high speed and low memory usage.**

| Feature / Concept | Purpose | Description |
|-------------------|----------|-------------|
| Histogram Binning (`max_bin=255`) | Faster split search | LightGBM converts continuous features into up to 255 bins by default and searches for splits on bin boundaries instead of every unique value, significantly reducing training time and memory usage. |
| Leaf-wise Tree Growth | Faster error reduction | Instead of expanding all nodes at the same depth, LightGBM always splits the leaf with the highest gain. This often produces more accurate trees with fewer splits but can increase overfitting if not controlled. |
| Native Categorical Features | Efficient handling of categories | LightGBM can directly consume categorical columns (pandas category dtype) and find optimal category groupings without requiring one-hot encoding. |
| Native Missing Value Handling | Automatic treatment of nulls | Missing values do not need imputation before training. During split finding, LightGBM learns whether missing values should go left or right based on which choice improves gain the most. |
| GOSS (Gradient-based One-Side Sampling) | Faster boosting | Retains rows with large prediction errors and samples rows with small errors. Focuses learning on the most informative observations while reducing computational cost.(top_rat & other_rate : percentage of high-gradient samples retained, and percentage of low-gradient samples retained) |
| EFB (Exclusive Feature Bundling) | Dimensionality reduction | Combines mutually exclusive sparse features into a smaller set of bundled features, reducing memory usage and accelerating training. |
| Gain Importance | Feature importance measurement | Measures how much a feature contributes to reducing the objective function across all splits. Usually the most informative feature-importance metric. |
| Split Importance | Feature importance measurement | Counts how many times a feature is used in splits across all trees. Useful but can overvalue features with many potential split points. |
| Monotonic Constraints | Enforce business rules | Allows users to specify that predictions must increase or decrease as certain features increase. Commonly used in risk, lending, and regulatory models. |
| `monotone_constraints` | Constraint specification | Uses values of 1 (increasing), -1 (decreasing), and 0 (unconstrained) for each feature to enforce monotonic relationships. |
| Early Stopping | Prevent overfitting | Stops training when validation performance no longer improves, preventing unnecessary trees from being added. |
| Histogram Reuse | Computational efficiency | LightGBM reuses previously computed histograms when evaluating child nodes, reducing repeated calculations and improving speed. |
| Categorical Split Optimization | Efficient category grouping | Instead of one-hot encoding, LightGBM searches for optimal category partitions based on target statistics and gain calculations. |
| `max_cat_to_onehot` | Category handling strategy | Small-cardinality categorical features may be internally treated similarly to one-hot encoding, while larger ones use LightGBM's categorical split algorithm. |
| Parallel Learning Support | Scalability | LightGBM supports distributed and parallel training, making it suitable for very large datasets. |
| GPU Training | Accelerated computation | Supports GPU-based histogram construction and split finding, substantially reducing training time for large-scale problems. |

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

In [2]:
# existing dataset : fraud (some columsn are not useful for the training)
df = pd.read_csv('data/development_data_preprocessed.csv')
drop_cols = ['fraud_tx_count_next_1h','history_first_tx_time','history_last_tx_time',
       'first_fraud_time_next_1h', 'fraud_amount_next_1h','last_tx_site_id',
       'minutes_to_first_fraud_next_1h','card_id', 'run_time', 'feature_cutoff_time', 'decision_time']

df = df[[i for i in df.columns if i not in drop_cols]]

In [3]:
df.shape

(19555, 79)

In [4]:
df.columns

Index(['history_tx_count_total', 'history_spend_total',
       'history_quantity_total', 'history_fraud_tx_count_total', 'tx_count_1h',
       'spend_1h', 'quantity_1h', 'tx_count_24h', 'spend_24h', 'quantity_24h',
       'avg_amount_24h', 'max_amount_24h', 'tx_count_7d', 'spend_7d',
       'quantity_7d', 'avg_amount_7d', 'tx_count_30d', 'spend_30d',
       'quantity_30d', 'avg_amount_30d', 'max_amount_30d', 'std_amount_30d',
       'avg_quantity_30d', 'cross_border_count_24h', 'cross_border_count_30d',
       'country_change_count_30d', 'quick_repeat_count_30d',
       'impossible_travel_150_count_30d', 'impossible_travel_300_count_30d',
       'night_tx_count_30d', 'weekend_tx_count_30d', 'magstripe_tx_count_30d',
       'mobile_tx_count_30d', 'chip_tx_count_30d', 'max_distance_km_30d',
       'avg_distance_km_30d', 'avg_gap_minutes_30d',
       'hours_since_first_transaction', 'minutes_since_last_tx',
       'hours_since_last_cross_border_tx', 'hours_since_last_fraud_tx',
       'la

In [5]:
df.isna().sum()[df.isna().sum()>0] # no nulls

Series([], dtype: int64)

In [6]:
# finding the categorical features: object type : converting to category type 
string_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()

In [7]:
for i in string_cols:
    df[i] = df[i].astype("category")

In [8]:
df.select_dtypes(include=['category']).columns.tolist()

['last_tx_site_country',
 'last_tx_product_description',
 'last_tx_processing_method']

In [9]:
from sklearn.model_selection import train_test_split

X = df.drop("target_next_1h", axis=1)
y = df["target_next_1h"]

# Test Split
X_temp, X_test, y_temp, y_test = train_test_split(
    X,y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# Validation Split
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp,y_temp,
    test_size=0.1765,
    random_state=42,
    stratify=y_temp
)

print(len(X_train))
print(len(X_valid))
print(len(X_test))

13687
2934
2934


In [10]:
print('train = ',dict(y_train.value_counts()))
print('valid = ',dict(y_valid.value_counts()))
print('test = ',dict(y_test.value_counts()))

train =  {0: np.int64(13633), 1: np.int64(54)}
valid =  {0: np.int64(2923), 1: np.int64(11)}
test =  {0: np.int64(2922), 1: np.int64(12)}


In [11]:
train_dataset = lgb.Dataset(X_train,label=y_train,categorical_feature=string_cols)
valid_dataset = lgb.Dataset(X_valid,label=y_valid,reference=train_dataset)
# reference : tells lgbm that the validation set should reuse metadata and feature binning from the train data 

In [12]:
def build_monotonic_constraints(n_features, constraint_map):
    constraints = np.zeros(n_features, dtype=int)
    for idx, value in constraint_map.items():
        constraints[idx] = value
    return constraints.tolist()

In [14]:
mono_constraints = build_monotonic_constraints(X.shape[1],constraint_map= {1:-1,2:1})

In [15]:
params = {
    "objective": "binary",      # binary/regression/multiclass/lambdarank
    "metric": "auc",            # rmse,mape,huber,quantile,binary,average_precision(area under pr-recall curve)
    "learning_rate": 0.03,      # Smaller updates per tree; usually generalizes better
    "num_iterations": 1000,     # More trees because learning rate is small
    "num_leaves": 64,           # Main complexity control; more leaves = more complex model - by default 31 
    "max_depth": 8,             # Prevents extremely deep trees - default is -1 ( no restriction ) 
    "min_data_in_leaf": 100,    # Minimum rows required in a leaf; strong overfitting control
    "min_gain_to_split": 0.05,  # Ignore weak splits
    "feature_fraction": 0.8,    # Use only 80% of columns per tree
    # "bagging_fraction": 0.8,    # Use only 80% of rows per tree
    # "bagging_freq": 1,          # Apply row sampling every iteration
    "lambda_l1": 0.5,           # L1 regularization
    "lambda_l2": 1.0,           # L2 regularization
    "cat_smooth": 20,           # Stabilizes rare categories : giving more trust towards the category with higher number of samples. 
    "min_data_per_group": 50,   # Each category there should be minimum 50 samples to be considered as safe/trust
    "is_unbalance": True,       # Auto-adjust class weights
    # "scale_pos_weight": 10
    "seed": 42,
    "boosting_type":"goss",     # GOSS (Gradient-based One-Side Sampling), lgbm cannot use bagging in GOSS 
    "verbosity": -1,
    "monotonic_constraints":mono_constraints
}

In [16]:
model = lgb.train(
    params=params,
    train_set=train_dataset,
    valid_sets=[train_dataset, valid_dataset],
    valid_names=["train", "valid"],
    num_boost_round=100, 
)

# CatBoost 

CatBoost is a gradient boosting framework that natively handles categorical features using ordered target statistics, reducing target leakage and eliminating most manual encoding requirements

| Feature / Concept | Purpose | Description |
|-------------------|----------|-------------|
| Ordered Target Statistics (CTR Features) | Leakage-safe categorical encoding | Converts categorical variables into smoothed target-based numerical features using permutations, ensuring each row’s encoding is computed without using its own label. |
| Multiple CTR Variants | Rich feature representation | Generates different statistics such as CTR(category), CTR(category + feature), frequency-based stats, and smoothed priors to capture diverse predictive signals. |
| Ordered Boosting | Reduces prediction shift | Computes gradients using models trained without the current sample (via permutations), reducing overfitting caused by training-on-own-errors bias. |
| Symmetric (Oblivious) Trees | Stable and fast model structure | Builds trees where all nodes at the same depth use the same split condition, resulting in balanced, cache-efficient, and highly regular tree structures. |
| Automatic Categorical Handling | No manual encoding required | Accepts raw categorical features and internally transforms them into CTR-based numeric representations without one-hot encoding or manual preprocessing. |
| High Cardinality Feature Handling | Scalable categorical learning | Handles large-category features efficiently by compressing them into statistically regularized CTR features instead of expanding them into sparse vectors. |
| Feature Combinations (CTR Cross Features) | Captures feature interactions | Automatically constructs interaction-based CTR features between categorical variables (e.g., merchant × device) to model conditional dependencies. |
| Prior-Based Smoothing | Prevents overfitting on rare categories | Blends category-level statistics with global priors to avoid overconfidence in categories with few observations. |
| Permutation-Based Training Strategy | Reduces leakage and bias | Uses multiple random permutations of training data to compute robust statistics and gradients, improving generalization. |
| Handling of Unseen Categories | Robust inference behavior | Assigns unseen categories at inference time to prior/global statistics or smoothed defaults instead of failing or encoding them arbitrarily. |
| Categorical Feature Combinations Limit Control | Prevents combinatorial explosion | Restricts and regularizes which feature combinations are generated to avoid exponential growth in sparse interaction features. |
| L2 Regularization on Leaf Values (`l2_leaf_reg`) | Controls overfitting | Penalizes large leaf predictions to stabilize model outputs, especially important when CTR features are strong. |
| Depth Control (`depth`) | Controls model complexity | Limits tree depth since CatBoost uses symmetric trees; replaces LightGBM’s `num_leaves` concept. |
| Random Strength | Adds stochastic regularization | Injects randomness into split selection to improve generalization and reduce overfitting on noisy datasets. |
| Bagging Temperature | Sampling control mechanism | Controls stochasticity of row sampling in boosting process, influencing variance vs bias trade-off. |
| Auto Class Weights | Handles class imbalance | Automatically adjusts class weights for imbalanced datasets like fraud detection without manual tuning. |
| Monotonic Constraints | Enforces business logic | Forces model predictions to increase or decrease with respect to selected features, improving interpretability and regulatory compliance. |
| Missing Value Handling | Native robustness to nulls | Learns optimal handling of missing values internally without requiring imputation or preprocessing. |
| Evaluation with Ordered Data Strategy | Stable validation behavior | Ensures evaluation metrics are computed in a way consistent with permutation-based training to avoid optimistic bias. |
| CPU-Optimized Symmetric Tree Inference | Fast prediction | Uses fixed-depth decision paths for efficient and highly cache-friendly inference, making deployment fast even on large models. |
| Native Support for Categorical Features | Simplified pipeline | Eliminates need for preprocessing steps like one-hot or label encoding by integrating categorical processing into training. |
| CTR Prior Strength Control | Controls smoothing intensity | Balances influence between category-specific statistics and global dataset prior to avoid overfitting on sparse categories. |

In [17]:
from catboost import CatBoostClassifier, Pool

In [18]:
cat_features = ['last_tx_site_country',
 'last_tx_product_description',
 'last_tx_processing_method'
]

train_pool = Pool(data=X_train,label=y_train,cat_features=cat_features)
valid_pool = Pool(data=X_valid,label=y_valid,cat_features=cat_features)

params = {
    "loss_function": "Logloss",        # binary classification objective
    "eval_metric": "AUC",              # same as LightGBM metric=auc
    "iterations": 100,                # equivalent to num_iterations
    "learning_rate": 0.03,             # same concept
    "depth": 8,                        # equivalent to max_depth (no num_leaves in CatBoost)
    "l2_leaf_reg": 3,                  # L2 regularization (CatBoost equivalent of lambda_l2)
                                       # L1 is not a primary knob in CatBoost like LGBM
    "random_strength": 1.0,            # adds randomness to split selection (regularization)
    "bagging_temperature": 1.0,        # stochastic row sampling control (instead of bagging_fraction/freq)
    "auto_class_weights": "Balanced",  # equivalent intent of is_unbalance=True
    "verbose": 0,
    "allow_writing_files": False,
    "random_seed": 42
}

In [19]:
model = CatBoostClassifier(**params)
model = model.fit(train_pool,eval_set=valid_pool,use_best_model=True)